# Mid Fusion Lite - Quick Test

Notebook đơn giản để test thay đổi feature engineering.
Chạy `Run All` để lấy kết quả CV pAUC.

In [ ]:
# ===== CONFIG =====
PRESET = 'quick'  # 'quick' cho 3 folds, 5 epochs
NUM_WORKERS = 0   # Windows compatibility

# Tắt heavy feature engineering để test nhanh
USE_CONTEXTUAL_GROUP_NORM_FEATURES = False
USE_UGLY_DUCKLING_FEATURES = False
USE_ANATOMICAL_ZSCORE_FEATURES = False
USE_AREA_AGGREGATION_FEATURES = False

# Data paths (sửa nếu cần)
TRAIN_METADATA_PATH = r"D:\HK1_2025_2026\SEMINAR_CODE\data\train-metadata.csv"
TEST_METADATA_PATH = r"D:\HK1_2025_2026\SEMINAR_CODE\data\test-metadata.csv"
SAMPLE_SUBMISSION_PATH = r"D:\HK1_2025_2026\SEMINAR_CODE\data\sample_submission.csv"
TRAIN_HDF5_PATH = r"D:\HK1_2025_2026\SEMINAR_CODE\data\train-image.hdf5"
TEST_HDF5_PATH = r"D:\HK1_2025_2026\SEMINAR_CODE\data\test-image.hdf5"
OUTPUT_DIR = './mid_fusion_lite_outputs/quick_test'

print(f"✅ Config loaded")
print(f"  Preset: {PRESET}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# Install dependencies first
import subprocess
import sys

print("Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", 
                        r"D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\requirements.txt"])
print("✅ Dependencies installed")

In [ ]:
import sys
from pathlib import Path

# Add code to path
code_root = Path(r'D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite')
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))

# Import and reload modules to get latest changes
import importlib
import config as config_module
import tabular as tabular_module
import runner as runner_module

config_module = importlib.reload(config_module)
tabular_module = importlib.reload(tabular_module)
runner_module = importlib.reload(runner_module)

print(f"✅ Modules imported")
print(f"  config: {Path(config_module.__file__).name}")
print(f"  tabular: {Path(tabular_module.__file__).name}")
print(f"  runner: {Path(runner_module.__file__).name}")

In [ ]:
from runner import run_with_preset

# Build overrides
overrides = {
    'output_dir': OUTPUT_DIR,
    'train_metadata_path': TRAIN_METADATA_PATH,
    'test_metadata_path': TEST_METADATA_PATH,
    'sample_submission_path': SAMPLE_SUBMISSION_PATH,
    'train_hdf5_path': TRAIN_HDF5_PATH,
    'test_hdf5_path': TEST_HDF5_PATH,
    'num_workers': NUM_WORKERS,
    'use_contextual_group_norm_features': USE_CONTEXTUAL_GROUP_NORM_FEATURES,
    'use_ugly_duckling_features': USE_UGLY_DUCKLING_FEATURES,
    'use_anatomical_zscore_features': USE_ANATOMICAL_ZSCORE_FEATURES,
    'use_area_aggregation_features': USE_AREA_AGGREGATION_FEATURES,
}

print(f"✅ Overrides prepared")
print(f"  Preset: {PRESET}")
print(f"  Feature flags: contextual={USE_CONTEXTUAL_GROUP_NORM_FEATURES}, ugly_duckling={USE_UGLY_DUCKLING_FEATURES}")

In [ ]:
print(f"\n🚀 Starting {PRESET} preset CV...\n")
print("="*80)

result = run_with_preset(PRESET, **overrides)

print("="*80)
print(f"\n✅ Training complete!\n")

In [ ]:
import pandas as pd
import json

print("\n" + "="*80)
print("📊 RESULTS SUMMARY")
print("="*80)

if result:
    print(f"\n✅ CV pAUC Score: {result.get('cv_pauc', 'N/A'):.6f}")
    print(f"✅ Submission file: {result.get('submission_path', 'N/A')}")
    print(f"✅ OOF predictions: {result.get('oof_path', 'N/A')}")
    print(f"✅ Metrics file: {result.get('metrics_path', 'N/A')}")
    
    # Print fold-wise metrics
    if 'fold_metrics' in result:
        print(f"\n📈 Per-Fold Metrics:")
        for i, metrics in enumerate(result['fold_metrics']):
            print(f"  Fold {i}: pAUC={metrics.get('pauc', 'N/A'):.6f}, AUC={metrics.get('auroc', 'N/A'):.6f}")
    
    # Display full result dict
    print(f"\n📦 Full Result:")
    for key, value in result.items():
        if not isinstance(value, list):
            print(f"  {key}: {value}")
else:
    print("❌ No result returned")

print("\n" + "="*80)

In [ ]:
# Save summary to file
from pathlib import Path
import json

output_path = Path(OUTPUT_DIR) / 'run_summary.txt'
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    f.write("MID FUSION LITE - QUICK TEST RESULTS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Preset: {PRESET}\n")
    f.write(f"Output Directory: {OUTPUT_DIR}\n\n")
    
    f.write("Feature Engineering Settings:\n")
    f.write(f"  - Contextual Group Norm: {USE_CONTEXTUAL_GROUP_NORM_FEATURES}\n")
    f.write(f"  - Ugly Duckling: {USE_UGLY_DUCKLING_FEATURES}\n")
    f.write(f"  - Anatomical Z-score: {USE_ANATOMICAL_ZSCORE_FEATURES}\n")
    f.write(f"  - Area Aggregation: {USE_AREA_AGGREGATION_FEATURES}\n\n")
    
    if result:
        f.write(f"CV pAUC Score: {result.get('cv_pauc', 'N/A'):.6f}\n")
        f.write(f"Submission: {result.get('submission_path', 'N/A')}\n")
        f.write(f"OOF Predictions: {result.get('oof_path', 'N/A')}\n")
        f.write(f"Metrics: {result.get('metrics_path', 'N/A')}\n")

print(f"✅ Summary saved to: {output_path}")